# L6-3a Demo: SAC vs PPO on the crawler

This notebook operationalizes the Lecture 3 claim that SAC is more sample-efficient on the same crawler and reward, while PPO can still be attractive on wall-clock grounds because it is operationally lighter. It reuses the shared `CrawlerEnv` and the same dense forward-velocity objective used in Lecture 2.

The lecture connection is:
- same environment as L6-2
- same reward for both agents
- action distribution comparison for the slide that contrasts PPO's boundary-seeking behavior with SAC's smoother stochastic policy

Lecture citation anchors: `haarnoja2018sac`, `raffin2024sacmassivelyparallel`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from l6_3_utils import (
    CMM_AMBER,
    CMM_BLUE,
    CMM_SLATE,
    FIGURE_DIR,
    rollout_policy,
    save_figure,
    smooth_xy,
    train_or_load,
)

TOTAL_TIMESTEPS = 50_000
SEED = 7
ENV_KWARGS = dict(
    include_velocity=True,
    action_mode="torque",
    reward_mode="dense_vel",
)

print(f"Saving lecture figures into: {FIGURE_DIR}")


In [ ]:
sac_artifact = train_or_load(
    "sac",
    "l6_3a_sac.zip",
    ENV_KWARGS,
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
)
ppo_artifact = train_or_load(
    "ppo",
    "l6_3a_ppo.zip",
    ENV_KWARGS,
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
)

print("SAC episodes logged:", len(sac_artifact.rewards))
print("PPO episodes logged:", len(ppo_artifact.rewards))
print("Recent SAC reward:", np.mean(sac_artifact.rewards[-5:]))
print("Recent PPO reward:", np.mean(ppo_artifact.rewards[-5:]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), sharey=True)

for ax, x_key, x_label, title in [
    (axes[0], "steps", "environment steps", "Sample axis"),
    (axes[1], "wallclock", "wall-clock seconds", "Wall-clock axis"),
]:
    if x_key == "steps":
        sac_x, sac_y = smooth_xy(sac_artifact.steps, sac_artifact.rewards, window=5)
        ppo_x, ppo_y = smooth_xy(ppo_artifact.steps, ppo_artifact.rewards, window=5)
    else:
        sac_x, sac_y = smooth_xy(sac_artifact.wallclock, sac_artifact.rewards, window=5)
        ppo_x, ppo_y = smooth_xy(ppo_artifact.wallclock, ppo_artifact.rewards, window=5)

    ax.plot(sac_x, sac_y, color=CMM_BLUE, lw=2.0, label="SAC")
    ax.plot(ppo_x, ppo_y, color=CMM_AMBER, lw=2.0, label="PPO")
    ax.set_xlabel(x_label)
    ax.set_title(title, color=CMM_SLATE)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="lower right")

axes[0].set_ylabel("episode reward")
curve_path = save_figure(fig, "sac_vs_ppo_curves.pdf")
curve_path


In [ ]:
sac_rollout = rollout_policy(sac_artifact.model, ENV_KWARGS, seed=SEED, max_steps=200)
ppo_rollout = rollout_policy(ppo_artifact.model, ENV_KWARGS, seed=SEED, max_steps=200)

fig, ax = plt.subplots(figsize=(4.6, 2.6))
bins = np.linspace(-1.0, 1.0, 31)
ax.hist(
    sac_rollout["actions"][:, 0],
    bins=bins,
    alpha=0.65,
    color=CMM_BLUE,
    label="SAC",
)
ax.hist(
    ppo_rollout["actions"][:, 0],
    bins=bins,
    alpha=0.55,
    color=CMM_AMBER,
    label="PPO",
)
ax.set_xlabel("action component value")
ax.set_ylabel("count")
ax.set_title("Deterministic rollout action histogram", color=CMM_SLATE)
ax.legend(loc="upper center")
hist_path = save_figure(fig, "sac_vs_ppo_action_dist.pdf")

print("SAC final x:", float(sac_rollout["x"][-1]))
print("PPO final x:", float(ppo_rollout["x"][-1]))
hist_path


On this toy crawler, SAC usually climbs faster per environment sample while PPO is cheaper per unit wall-clock work. The exact crossover point depends on the implementation budget and simulator throughput, which is the same practical caveat discussed in the lecture.
